<a href="https://colab.research.google.com/github/Omkekan/Auto-RAG-Agent-with-Hallucination-Guardrails/blob/main/self_correcting_rag_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autonomous RAG Agent with Hallucination Guardrails

A self-correcting RAG pipeline: retrieve -> grade relevance -> (re-query if weak) -> generate -> grade groundedness -> (regenerate if unsupported) -> grade answer relevance -> return.

Stack: LangChain + LangGraph, Qdrant (embedded, hybrid dense+sparse), Ollama (local), FastAPI.

Run cells top to bottom. First cell requires **Runtime > Change runtime type > T4 GPU**.

## Phase 1 - Environment setup

In [ ]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (569 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently i

In [ ]:
!pip install -q langchain langgraph langchain-community langchain-ollama \
    qdrant-client fastembed sentence-transformers \
    fastapi uvicorn pyngrok nest-asyncio datasets langchain-huggingface

# ragas>=0.4.0 has a known broken import (tries to load a ChatVertexAI path that
# no longer exists in langchain-community) - pinning to the last working release.
# See: https://github.com/vibrantlabsai/ragas/issues/2745
!pip install -q ragas==0.3.9

!curl -fsSL https://ollama.com/install.sh | sh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━

In [ ]:
import subprocess, time

# Ollama has no systemd in Colab, so run it as a background process
ollama_proc = subprocess.Popen(["ollama", "serve"])
time.sleep(5)

MODEL_NAME = "llama3.2:3b"  # swap for phi3:mini or qwen2.5:3b-instruct if you prefer
!ollama pull {MODEL_NAME}

## Phase 2 - Dataset

SQuAD 2.0 doubles as both your knowledge base (contexts) and your eval set (questions + answers, including unanswerable ones for testing the guardrail).

In [ ]:
from datasets import load_dataset

squad = load_dataset("rajpurkar/squad_v2", split="train[:3000]")

# knowledge base: unique passages
corpus = list(set(squad["context"]))
print(f"{len(corpus)} unique passages")

# eval set for Phase 7: mix of answerable + unanswerable questions
eval_set = [
    {
        "question": ex["question"],
        "answer": ex["answers"]["text"][0] if ex["answers"]["text"] else None,
        "context": ex["context"],
        "unanswerable": len(ex["answers"]["text"]) == 0,
    }
    for ex in squad.select(range(300))
]
print(f"{len(eval_set)} eval questions, {sum(e['unanswerable'] for e in eval_set)} unanswerable")

README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

squad_v2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

squad_v2/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 1.35MB            

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

368 unique passages
300 eval questions, 0 unanswerable


## Phase 3 - Chunk, embed, and index in Qdrant (hybrid dense + sparse)

In [ ]:
!pip install -qU langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)

chunks = []
for i, passage in enumerate(corpus):
    for j, piece in enumerate(splitter.split_text(passage)):
        chunks.append({"id": i * 100 + j, "text": piece})

print(f"{len(chunks)} chunks")

644 chunks


In [ ]:
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding

COLLECTION = "rag_docs"

client = QdrantClient(path="./qdrant_data")  # embedded mode, no server needed
dense_model = TextEmbedding("BAAI/bge-small-en-v1.5")
sparse_model = SparseTextEmbedding("Qdrant/bm25")

sample_dense = list(dense_model.embed(["test"]))[0]

client.recreate_collection(
    collection_name=COLLECTION,
    vectors_config={"dense": models.VectorParams(size=len(sample_dense), distance=models.Distance.COSINE)},
    sparse_vectors_config={"sparse": models.SparseVectorParams()},
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

/tmp/ipykernel_401/2311999923.py:12: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [ ]:
BATCH = 64
texts = [c["text"] for c in chunks]

for start in range(0, len(texts), BATCH):
    batch_texts = texts[start:start + BATCH]
    batch_chunks = chunks[start:start + BATCH]
    dense_vecs = list(dense_model.embed(batch_texts))
    sparse_vecs = list(sparse_model.embed(batch_texts))

    points = []
    for c, d, s in zip(batch_chunks, dense_vecs, sparse_vecs):
        points.append(models.PointStruct(
            id=c["id"],
            vector={
                "dense": d.tolist(),
                "sparse": models.SparseVector(indices=s.indices.tolist(), values=s.values.tolist()),
            },
            payload={"text": c["text"]},
        ))
    client.upsert(collection_name=COLLECTION, points=points)

print("Indexing done")

Indexing done


In [ ]:
def hybrid_search(query: str, k: int = 5):
    dense_vec = list(dense_model.embed([query]))[0]
    sparse_vec = list(sparse_model.embed([query]))[0]
    results = client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            models.Prefetch(query=dense_vec.tolist(), using="dense", limit=20),
            models.Prefetch(
                query=models.SparseVector(indices=sparse_vec.indices.tolist(), values=sparse_vec.values.tolist()),
                using="sparse", limit=20,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=k,
    )
    return [p.payload["text"] for p in results.points]

# quick sanity check
hybrid_search("What continent is Normandy in?")

["Spectre opened in Germany with $22.45 million (including previews), which included a new record for the biggest Saturday of all time, Australia with $8.7 million (including previews) and South Korea opened to $8.2 million (including previews). Despite the 13 November Paris attacks, which led to numerous theaters being closed down, the film opened with $14.6 million (including $2 million in previews) in France. In Mexico, where part of the film was shot, it debuted with more than double that of Skyfall with $4.5 million. It also bested its predecessor's opening in various Nordic regions where",
 'Filming started in Austria in December 2014, with production taking in the area around Sölden—including the Ötztal Glacier Road, Rettenbach glacier and the adjacent ski resort and cable car station—and Obertilliach and Lake Altaussee, before concluding in February 2015. Scenes filmed in Austria centred on the Ice Q Restaurant, standing in for the fictional Hoffler Klinik, a private medical cl

## Phase 4 - Graph nodes

Each node reads/writes a shared state dict. Grading nodes only *record* their verdict in state; separate router functions read that verdict to decide the next edge (this split keeps LangGraph happy).

In [ ]:
from typing import List, Optional, TypedDict
from langchain_ollama import ChatOllama

llm = ChatOllama(model=MODEL_NAME, temperature=0)

class GraphState(TypedDict):
    question: str
    documents: List[str]
    generation: str
    transform_retries: int
    hallucination_retries: int
    grounded: Optional[bool]
    answer_relevant: Optional[bool]

In [ ]:
def retrieve(state: GraphState) -> GraphState:
    docs = hybrid_search(state["question"])
    return {**state, "documents": docs}

In [ ]:
GRADE_PROMPT = """You are grading whether a retrieved passage is relevant to a question.
Question: {question}
Passage: {document}
Reply with exactly one word: yes or no."""

def grade_documents(state: GraphState) -> GraphState:
    question = state["question"]
    kept = []
    for doc in state["documents"]:
        resp = llm.invoke(GRADE_PROMPT.format(question=question, document=doc))
        if "yes" in resp.content.strip().lower():
            kept.append(doc)
    return {**state, "documents": kept}

def decide_to_generate(state: GraphState) -> str:
    if len(state["documents"]) == 0 and state.get("transform_retries", 0) < 2:
        return "transform_query"
    return "generate"

In [ ]:
def transform_query(state: GraphState) -> GraphState:
    original = state["question"]
    resp = llm.invoke(
        "Rewrite this question to be more specific and easier for a search engine "
        f"to retrieve relevant passages for: {original}\nRewritten question:"
    )
    return {
        **state,
        "question": resp.content.strip(),
        "transform_retries": state.get("transform_retries", 0) + 1,
    }

In [ ]:
GEN_PROMPT = """Answer the question using ONLY the context below.
If the context does not contain the answer, say exactly: "I don't have enough information to answer that."

Context:
{context}

Question: {question}
Answer:"""

def generate(state: GraphState) -> GraphState:
    context = "\n\n".join(state["documents"]) if state["documents"] else "(no relevant context found)"
    resp = llm.invoke(GEN_PROMPT.format(context=context, question=state["question"]))
    return {**state, "generation": resp.content.strip()}

In [ ]:
HALLUCINATION_PROMPT = """Facts:
{documents}

Answer:
{generation}

Is the answer fully supported by the facts above, with no invented claims? Reply with exactly one word: yes or no."""

def grade_hallucination(state: GraphState) -> GraphState:
    resp = llm.invoke(HALLUCINATION_PROMPT.format(
        documents="\n\n".join(state["documents"]) or "(none)",
        generation=state["generation"],
    ))
    grounded = "yes" in resp.content.strip().lower()
    retries = state.get("hallucination_retries", 0) + (0 if grounded else 1)
    return {**state, "grounded": grounded, "hallucination_retries": retries}

def route_after_hallucination_check(state: GraphState) -> str:
    if state["grounded"]:
        return "grade_answer_relevance"
    if state.get("hallucination_retries", 0) < 2:
        return "generate"
    return "fallback"

In [ ]:
RELEVANCE_PROMPT = """Question: {question}
Answer: {generation}
Does the answer directly and completely address the question? Reply with exactly one word: yes or no."""

def grade_answer_relevance(state: GraphState) -> GraphState:
    resp = llm.invoke(RELEVANCE_PROMPT.format(question=state["question"], generation=state["generation"]))
    relevant = "yes" in resp.content.strip().lower()
    return {**state, "answer_relevant": relevant}

def route_after_relevance_check(state: GraphState) -> str:
    if state["answer_relevant"]:
        return "end"
    if state.get("transform_retries", 0) < 2:
        return "transform_query"
    return "fallback"

def fallback(state: GraphState) -> GraphState:
    return {**state, "generation": "I don't have enough grounded information in the knowledge base to answer that confidently."}

## Phase 5 - Wire the LangGraph state machine

In [ ]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(GraphState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("transform_query", transform_query)
workflow.add_node("generate", generate)
workflow.add_node("grade_hallucination", grade_hallucination)
workflow.add_node("grade_answer_relevance", grade_answer_relevance)
workflow.add_node("fallback", fallback)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges("grade_documents", decide_to_generate, {
    "transform_query": "transform_query",
    "generate": "generate",
})
workflow.add_edge("transform_query", "retrieve")
workflow.add_edge("generate", "grade_hallucination")
workflow.add_conditional_edges("grade_hallucination", route_after_hallucination_check, {
    "generate": "generate",
    "grade_answer_relevance": "grade_answer_relevance",
    "fallback": "fallback",
})
workflow.add_conditional_edges("grade_answer_relevance", route_after_relevance_check, {
    "end": END,
    "transform_query": "transform_query",
    "fallback": "fallback",
})
workflow.add_edge("fallback", END)

app = workflow.compile()

In [ ]:
result = app.invoke({
    "question": "What continent is Normandy located on?",
    "documents": [],
    "generation": "",
    "transform_retries": 0,
    "hallucination_retries": 0,
    "grounded": None,
    "answer_relevant": None,
})
print(result["generation"])

I don't have enough information to answer that.


## Phase 6 - Serve behind FastAPI (with an ngrok tunnel, since Colab has no public IP)

Requires a free ngrok account + authtoken: https://dashboard.ngrok.com/get-started/your-authtoken

## Phase 7 - Measure the hallucination reduction (for your resume claim)

Compares a naive baseline RAG (retrieve top-k, generate, no grading) against this self-correcting pipeline on the SQuAD eval set, using RAGAS faithfulness and answer relevancy. Run this on a subsample first (20-30 questions) since each question costs several LLM calls in the guarded pipeline.

In [ ]:
def naive_rag(question: str) -> tuple[str, list[str]]:
    docs = hybrid_search(question, k=5)
    context = "\n\n".join(docs) if docs else "(no context)"
    resp = llm.invoke(GEN_PROMPT.format(context=context, question=question))
    return resp.content.strip(), docs

sample = eval_set[:25]  # widen once you confirm everything runs end to end

baseline_rows, guarded_rows = [], []
for ex in sample:
    q = ex["question"]

    b_answer, b_docs = naive_rag(q)
    baseline_rows.append({"question": q, "answer": b_answer, "contexts": b_docs})

    g_result = app.invoke({
        "question": q, "documents": [], "generation": "",
        "transform_retries": 0, "hallucination_retries": 0,
        "grounded": None, "answer_relevant": None,
    })
    guarded_rows.append({"question": q, "answer": g_result["generation"], "contexts": g_result["documents"]})

In [ ]:
import sys, types

stub = types.ModuleType("langchain_community.chat_models.vertexai")
stub.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = stub

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy


def to_ragas_dataset(rows):
    return Dataset.from_list([
        {"question": r["question"], "answer": r["answer"], "contexts": r["contexts"] or [""]}
        for r in rows
    ])

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.run_config import RunConfig

ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5"))
run_config = RunConfig(timeout=300, max_workers=2)

baseline_scores = evaluate(
    to_ragas_dataset(baseline_rows), metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm, embeddings=ragas_embeddings, run_config=run_config,
)
guarded_scores = evaluate(
    to_ragas_dataset(guarded_rows), metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm, embeddings=ragas_embeddings, run_config=run_config,
)

print("Baseline (naive RAG):", baseline_scores)
print("Self-correcting pipeline:", guarded_scores)

baseline_faith = baseline_scores["faithfulness"]
guarded_faith = guarded_scores["faithfulness"]
reduction = (baseline_faith - guarded_faith) / baseline_faith if baseline_faith else 0
print(f"\nFaithfulness improved from {baseline_faith:.2f} to {guarded_faith:.2f}")

/tmp/ipykernel_401/3062845032.py:17: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_401/3062845032.py:18: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5"))


Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[1]: OutputParserException(Invalid json output: Where was Beyonce born in the late 1990s?
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
ERROR:ragas.executor:Exception raised in Job[2]: OutputParserException(Invalid json output: Here is the analysis of the complexity of each sentence in the answer:

{
    "statements": [
        {
            "text": "She competed in various singing and dancing competitions as a child",
            "parts": [
                "She"
            ]
        },
        {
            "text": "specifically in Houston, Texas",
            "parts": []
        }
    ]
}

Here's how I broke down the sentences:

1. The first sentence contains a pronoun ("She"), so it cannot be broken down into fully understandable statements without using pronouns.
2. The second sentence does not contain any pronouns and can be broken down into two fully understandable statement

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[2]: OutputParserException(Invalid json output: def analyze_complexity(answer):
    statements = []
    for sentence in answer.split('.'):
        if sentence.strip():
            words = sentence.split()
            subject = None
            verb = None
            object = None
            preposition = None
            for word in words:
                if word.lower() in ['was', 'is', 'were', 'are']:
                    if subject is not None and verb is not None:
                        statements.append(f'"{subject} {verb} {object}."')
                    subject = word
                elif word.lower() in ['and', 'but', 'or', 'for']:
                    continue
                elif word.lower() in ['to', 'of', 'in', 'with']:
                    preposition = word
                else:
                    if verb is None:
                        verb = word
                    elif object is None:
                        object = word

KeyboardInterrupt: 

In [ ]:
#!pip show ragas | grep Version